# 프로젝트 : 얼굴을 인식하여 캐릭터 씌우기

## 패키지 설치
pip install mediapipe

In [11]:
import cv2
import mediapipe as mp

def overlay(image, x, y, w, h, overlay_image):
    """투명 PNG를 원본 이미지 위에 오버레이하는 함수"""
    
    # 원본 이미지 크기 확인
    h_img, w_img, _ = image.shape  

    # 오버레이 이미지 크기 조정 (원하는 w, h 크기로 변경)
    overlay_resized = cv2.resize(overlay_image, (w, h), interpolation=cv2.INTER_AREA)

    # 알파 채널 분리
    if overlay_resized.shape[2] == 4:  # 4채널 BGRA 이미지인 경우
        alpha = overlay_resized[:, :, 3] / 255.0  # 0~1 사이 값
        overlay_resized = overlay_resized[:, :, :3]  # BGR 채널만 남김
    else:
        alpha = np.ones((h, w), dtype=np.float32)  # 알파 채널이 없으면 불투명 처리

    # 좌표 계산 (이미지 크기 내로 제한)
    x1, x2 = max(0, x - w // 2), min(w_img, x + w // 2)
    y1, y2 = max(0, y - h // 2), min(h_img, y + h // 2)

    # 크기 조정 후, 좌표에 맞게 다시 슬라이싱
    overlay_resized = overlay_resized[: y2 - y1, : x2 - x1]
    alpha = alpha[: y2 - y1, : x2 - x1]

    # 오버레이 이미지 적용
    for c in range(3):  # BGR 채널
        image[y1:y2, x1:x2, c] = (
            overlay_resized[:, :, c] * alpha + image[y1:y2, x1:x2, c] * (1 - alpha)
        )
# 얼굴을 찾고, 찾은 얼굴에 표시를 해주기 위한 변수 정의
mp_face_detection = mp.solutions.face_detection # 얼굴 검출을 위한 face_detection 모듈을 사용
mp_drawing = mp.solutions.drawing_utils # 얼굴의 특징을 그리기 위한 drawing_utils 모뮬을 사용


# 동영상 파일 열기
cap = cv2.VideoCapture('face_video2.mp4')

fourcc = cv2.VideoWriter_fourcc(*'mp4v') # 코덱 설정
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
out = cv2.VideoWriter('result_output.mp4', fourcc, fps, (width, height))

# 이미지 불러오기
image_right_eye = cv2.imread('right_eye.png',cv2.IMREAD_UNCHANGED) # 200 * 300
image_left_eye = cv2.imread('left_eye.png',cv2.IMREAD_UNCHANGED) # 200 * 300
image_nose = cv2.imread('nose.png',cv2.IMREAD_UNCHANGED) # 100 * 200 (가로, 세로)

with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.8) as face_detection:
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            break

        # To improve performance, optionally mark the image as not writeable to
        # pass by reference.
        image.flags.writeable = False
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_detection.process(image)

        # Draw the face detection annotations on the image.
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        if results.detections:
            # 6개의 특징: 오른쪽 눈, 왼쪽 눈, 코 끝부분, 입 중심, 오른쪽 귀, 왼쪽 귀 (귀 구슬점, 이주)
            for detection in results.detections:
                
                # mp_drawing.draw_detection(image, detection)
                # print(detection)
                
                # 특정 위치 가져오기
                keypoints = detection.location_data.relative_keypoints
                right_eye = keypoints[0] # 오른쪽 눈
                left_eye = keypoints[1] # 왼쪽 눈
                nose_tip = keypoints[2] # 코 끝부분
                
                
                h, w, _ = image.shape # height, width, channel : 이미지로 부터 세로, 가로 크기 가져옴
                right_eye = (int(right_eye.x * w) - 40, int(right_eye.y * h) - 100) 
                left_eye = (int(left_eye.x * w) + 40, int(left_eye.y * h) - 100)
                nose_tip = (int(nose_tip.x * w), int(nose_tip.y * h))
                
                bbox = detection.location_data.relative_bounding_box
                bbox_w, bbox_h = int(bbox.width * w), int(bbox.height * h)

                # 얼굴 크기에 맞게 이미지 크기 조정
                eye_w, eye_h = bbox_w // 4, bbox_h // 4
                nose_w, nose_h = bbox_w // 3, bbox_h // 3

                overlay(image, *right_eye, eye_w, eye_h, image_right_eye)
                overlay(image, *left_eye, eye_w, eye_h, image_left_eye)
                overlay(image, *nose_tip, nose_w, nose_h, image_nose)
                
        out.write(image)
        
        # Flip the image horizontally for a selfie-view display.
        cv2.imshow('MediaPipe Face Detection', cv2.resize(image, None, fx = 0.5, fy = 0.5))
        if cv2.waitKey(1) == ord('q'):
            break
cap.release()
out.release()
cv2.destroyAllWindows()
print("영상 저장 완료! result_output.mp4")

영상 저장 완료! result_output.mp4
